# 🧠 nanoGPT — Build a GPT from Scratch
### *Andrej Karpathy style — every single line explained*

> *"The most important single idea in all of deep learning is that of a language model."*  
> — Andrej Karpathy

This notebook builds a **complete, working GPT** character-by-character from zero lines of code.  
No magic. No black boxes. Every tensor, every operation, every design decision is explained.

---
## 📚 What We Build

```
Phase 1 — Foundation
  ├── Tokenization (char-level)
  ├── Bigram Language Model (baseline)
  └── Why it fails → motivation for attention

Phase 2 — Self-Attention (the core)
  ├── The mathematical trick: lower-triangular averaging
  ├── Query, Key, Value explained from first principles
  ├── Scaled dot-product attention
  ├── Causal masking (why GPT can't see the future)
  └── Multi-head attention

Phase 3 — Full Transformer
  ├── Feed-Forward Network
  ├── Residual connections
  ├── Layer Normalization
  ├── Positional embeddings
  └── Complete GPT architecture

Phase 4 — Training & Generation
  ├── AdamW + gradient clipping
  ├── Learning rate schedule
  ├── Loss curves + perplexity
  ├── Temperature, top-k, nucleus sampling
  └── Attention pattern visualization

Phase 5 — Deep Dives
  ├── Scaling laws
  ├── GPT vs BERT architecture
  ├── Modern tricks: Flash Attention, RoPE, GQA
  └── How GPT-2 → GPT-4 differs from nanoGPT
```
---

---
# 🔷 PHASE 1 — Foundation: Data, Tokenization, Bigram Baseline
---

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'matplotlib',
                'numpy', '--quiet'], check=False)
print('✅ Ready!')

In [ ]:
import os, math, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility
torch.manual_seed(1337)   # Karpathy's favourite seed
np.random.seed(1337)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}')

## 1.1 The Dataset — Tiny Shakespeare

We use ~1MB of Shakespeare's complete works — the same file Karpathy uses in his lecture.

**Why Shakespeare?**  
- Small enough to train in minutes on CPU  
- Rich enough vocabulary to see real patterns  
- Character-level → tiny vocab (65 chars)  

**Character-level tokenization:**  
We map every unique character → integer. No BPE, no WordPiece — just raw characters.  
GPT-4 uses ~100K BPE tokens; we use 65 characters. Same principle, different scale.

In [ ]:
# ── Download Tiny Shakespeare ─────────────────────────────────
URL  = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
PATH = '/tmp/shakespeare.txt'

if not os.path.exists(PATH):
    print('Downloading Tiny Shakespeare...')
    urllib.request.urlretrieve(URL, PATH)

with open(PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Dataset: {len(text):,} characters')
print(f'\nFirst 500 characters:')
print(text[:500])

In [ ]:
# ── Character-level vocabulary ────────────────────────────────
#
# CONCEPT: A tokenizer maps between text and integers.
# Every LLM needs one. GPT-4 uses tiktoken (BPE, ~100K vocab).
# We use the simplest possible: one integer per unique character.

chars      = sorted(set(text))          # all unique characters
VOCAB_SIZE = len(chars)

# Build lookup tables
stoi = {ch: i  for i, ch in enumerate(chars)}   # char  → int
itos = {i:  ch for i, ch in enumerate(chars)}   # int   → char

# Encoder / decoder functions
encode = lambda s: [stoi[c] for c in s]           # str  → List[int]
decode = lambda l: ''.join(itos[i] for i in l)    # List[int] → str

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'Characters      : {repr("".join(chars))}')
print()

# Sanity check
sample = 'Hello, GPT!'
encoded = encode(sample)
decoded = decode(encoded)
print(f'Encode({repr(sample)}) → {encoded}')
print(f'Decode({encoded})      → {repr(decoded)}')
assert decoded == sample, 'Encode/decode roundtrip failed!'
print('✅ Encode/decode roundtrip OK')

# ── Visualise character frequency ─────────────────────────────
freq  = Counter(text)
top30 = sorted(freq.items(), key=lambda x: -x[1])[:30]

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].bar([repr(c) for c,_ in top30], [n for _,n in top30], color='steelblue', edgecolor='white')
axes[0].set_title('Top-30 Character Frequencies', fontweight='bold')
axes[0].set_xlabel('Character'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Token ID mapping visualisation
axes[1].imshow([[stoi[c] for c in chars]], cmap='tab20', aspect='auto')
axes[1].set_xticks(range(len(chars)))
axes[1].set_xticklabels([repr(c) for c in chars], rotation=90, fontsize=7)
axes[1].set_yticks([])
axes[1].set_title('Character → Token ID Mapping', fontweight='bold')

plt.tight_layout(); plt.show()

In [ ]:
# ── Encode entire dataset + train/val split ───────────────────
#
# IMPORTANT: We split BEFORE any processing.
# The val set must be data the model has NEVER seen.
# 90% train, 10% val — same ratio Karpathy uses.

data    = torch.tensor(encode(text), dtype=torch.long)
n       = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f'Total tokens   : {len(data):,}')
print(f'Train tokens   : {len(train_data):,}  ({len(train_data)/len(data)*100:.0f}%)')
print(f'Val tokens     : {len(val_data):,}  ({len(val_data)/len(data)*100:.0f}%)')
print(f'\nFirst 50 encoded tokens  : {data[:50].tolist()}')
print(f'Decoded back             : {repr(decode(data[:50].tolist()))}')

## 1.2 Hyperparameters

These control the **size** and **training** of our model.  
Karpathy's full model (GPU, 256 context, 384 embd, 6 layers, 6 heads) gets loss ~1.48.  
Our CPU-friendly version gets ~1.7–1.9 — still clearly reads like Shakespeare!

| Param | Ours (CPU) | Karpathy full (GPU) | GPT-2 Small |
|---|---|---|---|
| `BLOCK_SIZE` | 64 | 256 | 1024 |
| `N_EMBD` | 128 | 384 | 768 |
| `N_HEAD` | 4 | 6 | 12 |
| `N_LAYER` | 4 | 6 | 12 |
| `BATCH_SIZE` | 32 | 64 | — |
| **Params** | ~400K | ~10M | 124M |

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────
# Bump these up if you have a GPU for much better results!

BLOCK_SIZE    = 64      # max context length (sequence length T)
BATCH_SIZE    = 32      # sequences processed in parallel
N_EMBD        = 128     # embedding dimension (d_model)
N_HEAD        = 4       # number of attention heads
N_LAYER       = 4       # number of transformer blocks
DROPOUT       = 0.1     # dropout probability (0 = no dropout)
MAX_ITERS     = 5000    # training steps
EVAL_INTERVAL = 500     # evaluate every N steps
EVAL_ITERS    = 100     # batches to average for evaluation
LEARNING_RATE = 3e-4    # peak learning rate

# Derived
HEAD_SIZE = N_EMBD // N_HEAD

print('Model Configuration:')
print(f'  Block size   (T) : {BLOCK_SIZE}')
print(f'  Embedding dim(C) : {N_EMBD}')
print(f'  Heads            : {N_HEAD} × head_size={HEAD_SIZE}')
print(f'  Layers           : {N_LAYER}')
print(f'  Batch size   (B) : {BATCH_SIZE}')
print(f'  Max iters        : {MAX_ITERS:,}')
print(f'  Device           : {DEVICE}')

# Approximate param count before building model
approx = (VOCAB_SIZE*N_EMBD + BLOCK_SIZE*N_EMBD +
           N_LAYER * (3*N_EMBD*N_EMBD + N_EMBD*N_EMBD +
                      2*N_EMBD*4*N_EMBD + N_EMBD))
print(f'  Est. parameters  : ~{approx/1e6:.2f}M')

In [ ]:
# ── Batch loading ─────────────────────────────────────────────
#
# KEY INSIGHT: Every position in a sequence is a training example.
# Given context [w1, w2, ..., wt], predict wt+1.
# So from a single sequence of length T, we get T training examples!
# One batch of (B, T) gives us B*T = 32*64 = 2048 examples.

def get_batch(split):
    """
    Returns (x, y) where:
      x: (BATCH_SIZE, BLOCK_SIZE)  — input token sequences
      y: (BATCH_SIZE, BLOCK_SIZE)  — target token sequences (x shifted right by 1)

    For each position t in [0, BLOCK_SIZE):
      - input  = x[:, t]   = token at position t
      - target = y[:, t]   = token at position t+1  ← this is what we predict
    """
    data = train_data if split == 'train' else val_data
    # Pick BATCH_SIZE random start positions
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x  = torch.stack([data[i   : i+BLOCK_SIZE  ] for i in ix])   # (B, T)
    y  = torch.stack([data[i+1 : i+BLOCK_SIZE+1] for i in ix])   # (B, T)
    return x.to(DEVICE), y.to(DEVICE)


# Demonstrate what a batch looks like
xb, yb = get_batch('train')
print(f'x shape: {xb.shape}   (batch={BATCH_SIZE}, time={BLOCK_SIZE})')
print(f'y shape: {yb.shape}')

print(f'\nFirst sequence (raw token IDs):')
print(f'  x[0] = {xb[0].tolist()}')
print(f'  y[0] = {yb[0].tolist()}')

print(f'\nDecoded x[0]: {repr(decode(xb[0].tolist()))}')
print(f'Decoded y[0]: {repr(decode(yb[0].tolist()))}')

print('\nThe model sees each of these (context→target) pairs:')
for t in range(5):
    ctx    = decode(xb[0, :t+1].tolist())
    target = decode([yb[0, t].item()])
    print(f'  context={repr(ctx):20s}  →  target={repr(target)}')

## 1.3 Bigram Baseline — The Dumbest Possible Language Model

Before GPT, let's build the simplest possible language model: **bigram**.

**Idea:** Predict the next character based ONLY on the current character.  
No history. No context. Just: "given I see 'H', what comes next?"

**Implementation:** A single embedding table of shape `(vocab_size, vocab_size)`.  
Row `i` = logits for what comes after character `i`.

**Why it fails:** English (and Shakespeare!) has long-range dependencies.  
"The cat sat on the ___" — you need 6 words of context to predict "mat".  
A bigram can only use the last word.

**Loss floor:** Random guessing → loss = `log(vocab_size)` = `log(65)` ≈ **4.17**.  
Bigram → ~**2.5**. Good GPT → ~**1.5**.

In [ ]:
# ── Evaluation helper (used for all models) ───────────────────
@torch.no_grad()
def estimate_loss(model):
    """Average loss over EVAL_ITERS batches for train and val splits."""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


class BigramLM(nn.Module):
    """
    Bigram Language Model — simplest possible LM.

    token_embedding_table[i] = logits for what token comes after token i.
    NO context — each prediction depends ONLY on current token.
    """

    def __init__(self):
        super().__init__()
        # Shape: (vocab_size, vocab_size)
        # Each row i = unnormalised log-probabilities (logits) for next token
        self.table = nn.Embedding(VOCAB_SIZE, VOCAB_SIZE)

    def forward(self, idx, targets=None):
        # idx: (B, T)  — input token IDs
        logits = self.table(idx)    # (B, T, VOCAB_SIZE)

        if targets is None:
            return logits, None

        # Cross-entropy expects (N, C) and (N,)
        B, T, C = logits.shape
        loss = F.cross_entropy(
            logits.view(B*T, C),   # (B*T, VOCAB_SIZE)
            targets.view(B*T)      # (B*T,)
        )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Auto-regressive generation — one token at a time."""
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -BLOCK_SIZE:])   # (B, T, V)
            logits    = logits[:, -1, :] / temperature  # last step: (B, V)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1:]] = -float('inf')
            probs  = F.softmax(logits, dim=-1)
            next_t = torch.multinomial(probs, 1)       # (B, 1)
            idx    = torch.cat([idx, next_t], dim=1)   # (B, T+1)
        return idx


# ── Train bigram ──────────────────────────────────────────────
bigram = BigramLM().to(DEVICE)
print(f'Bigram params: {sum(p.numel() for p in bigram.parameters()):,}')
print(f'Random-guess loss: {math.log(VOCAB_SIZE):.4f}   (ceiling)')

opt = torch.optim.AdamW(bigram.parameters(), lr=1e-3)
bigram_losses = {'train': [], 'val': [], 'step': []}

for step in range(2000):
    x, y = get_batch('train')
    _, loss = bigram(x, y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 400 == 0:
        ls = estimate_loss(bigram)
        bigram_losses['train'].append(ls['train'])
        bigram_losses['val'].append(ls['val'])
        bigram_losses['step'].append(step)
        print(f'  step {step:4d} | train={ls["train"]:.4f} | val={ls["val"]:.4f}')

print(f'\nBigram final val loss: {bigram_losses["val"][-1]:.4f}')

# ── Sample from bigram ─────────────────────────────────────────
ctx = torch.zeros((1,1), dtype=torch.long, device=DEVICE)
bg_out = decode(bigram.generate(ctx, 300)[0].tolist())
print(f'\n--- Bigram Output (complete gibberish — no context!) ---\n{bg_out}')

---
# 🔷 PHASE 2 — Self-Attention: The Heart of GPT
---

The bigram is terrible because it has **zero memory** — it ignores all previous tokens.  
We need a mechanism that lets tokens **communicate** with each other.

Self-attention is that mechanism. We build it in **4 stages**, adding one idea at a time.

In [ ]:
# ============================================================
#  SELF-ATTENTION — Built in 4 Stages
#
#  STAGE 1: Naive average (loop version)
#  STAGE 2: Same result, matrix multiply (fast version)
#  STAGE 3: Weighted average with softmax + masking
#  STAGE 4: True self-attention with learned Q, K, V
# ============================================================

torch.manual_seed(1337)
B, T, C = 4, 8, 16    # batch, time, channels
x = torch.randn(B, T, C)

# ─────────────────────────────────────────────────────────────
# STAGE 1 — Naive averaging loop
# Each token at position t = average of all tokens from 0..t
# This is 'bag-of-words': each token 'sees' all past context.
# Problem: equal weight to ALL past tokens (recent and distant)
# ─────────────────────────────────────────────────────────────
print('=== STAGE 1: Naive averaging (manual loop) ===')

xbow = torch.zeros_like(x)   # 'bag of words'
for b in range(B):
    for t in range(T):
        xprev      = x[b, :t+1]       # all tokens 0..t  shape: (t+1, C)
        xbow[b, t] = xprev.mean(dim=0) # simple mean      shape: (C,)

print(f'x[0,0]   = {x[0,0,:5].tolist()}  (original token 0)')
print(f'xbow[0,0]= {xbow[0,0,:5].tolist()} (same — only 1 token in context)')
print(f'xbow[0,3]= {xbow[0,3,:5].tolist()} (mean of tokens 0-3)')

# ─────────────────────────────────────────────────────────────
# STAGE 2 — Same result using matrix multiplication (fast!)
# Lower-triangular matrix W:
#   W[t, t'] = 1/(t+1) if t' <= t, else 0
# So:  xbow = W @ x   achieves the same averaging.
# This is the KEY TRICK: averaging-over-past ≡ lower-tril matmul
# ─────────────────────────────────────────────────────────────
print('\n=== STAGE 2: Lower-triangular matrix trick ===')

tril = torch.tril(torch.ones(T, T))   # shape (T, T)
W    = tril / tril.sum(dim=1, keepdim=True)  # normalize rows

print(f'Weight matrix W (rows sum to 1):')
print(W.round(decimals=2))

xbow2 = W @ x   # (T, T) @ (B, T, C) → (B, T, C)  via broadcasting
print(f'\nMax diff Stage1 vs Stage2: {(xbow2 - xbow).abs().max():.1e}  ← should be ~0')

# ─────────────────────────────────────────────────────────────
# STAGE 3 — Replace division with softmax (same uniform weights,
# but now the framework for LEARNED weights is clear)
# The -inf mask ensures softmax gives 0 weight to future tokens
# ─────────────────────────────────────────────────────────────
print('\n=== STAGE 3: Softmax + causal mask ===')

W3   = torch.zeros(T, T)
W3   = W3.masked_fill(tril == 0, float('-inf'))   # future → -inf
W3   = F.softmax(W3, dim=-1)                      # -inf → 0 after softmax

print(f'W3 (after masking + softmax):')
print(W3.round(decimals=2))

xbow3 = W3 @ x
print(f'\nMax diff Stage2 vs Stage3: {(xbow3 - xbow2).abs().max():.1e}  ← should be ~0')
print('\nKey: replacing the fixed W with LEARNED weights = self-attention!')

In [ ]:
# ─────────────────────────────────────────────────────────────
# STAGE 4 — TRUE SELF-ATTENTION with Q, K, V
#
# Instead of uniform weights, each token ASKS what it needs (Q)
# and BROADCASTS what it contains (K).
# The weight between token i and token j =
#   how well token j's key matches token i's query.
#
# Q = query  — 'what am I looking for?'
# K = key    — 'what do I have?'
# V = value  — 'what do I actually communicate?'
#
# Attention formula:
#   A = softmax( Q·Kᵀ / √d_k  +  causal_mask ) · V
#
# The √d_k scaling prevents the dot products from growing too
# large (which would cause softmax to saturate into one-hot,
# making gradients vanish)
# ─────────────────────────────────────────────────────────────
print('=== STAGE 4: True self-attention with Q, K, V ===')

head_size = 16

# Learned linear projections (no bias — Karpathy convention)
Wq = nn.Linear(C, head_size, bias=False)   # projects to query space
Wk = nn.Linear(C, head_size, bias=False)   # projects to key space
Wv = nn.Linear(C, head_size, bias=False)   # projects to value space

q = Wq(x)   # (B, T, head_size) — each token's query vector
k = Wk(x)   # (B, T, head_size) — each token's key vector
v = Wv(x)   # (B, T, head_size) — each token's value vector

print(f'Q: {q.shape}   K: {k.shape}   V: {v.shape}')

# ── Step 1: Compute scaled dot-product attention scores ───────
scale = head_size ** -0.5            # 1/√head_size
attn  = q @ k.transpose(-2, -1)     # (B, T, T)
attn  = attn * scale                 # scale by 1/√d_k

print(f'\nUnscaled attn scores std : {(q @ k.transpose(-2,-1)).std():.4f}')
print(f'Scaled   attn scores std : {attn.std():.4f}  ← ~1.0 is what we want')
print(f'(Without scaling, softmax saturates to one-hot → vanishing gradients)')

# ── Step 2: Causal mask — tokens cannot attend to future ──────
mask = torch.tril(torch.ones(T, T, device=x.device))
attn = attn.masked_fill(mask == 0, float('-inf'))

# ── Step 3: Softmax → attention weights ──────────────────────
attn = F.softmax(attn, dim=-1)   # (B, T, T)  rows sum to 1

# ── Step 4: Weighted sum of values ───────────────────────────
out  = attn @ v                  # (B, T, head_size)

print(f'\nAttention weights for batch[0] (8×8 matrix, rows sum to 1):')
print(attn[0].detach().round(decimals=3))
print(f'\nOutput shape: {out.shape}')
print('Output = weighted average of VALUE vectors, weighted by query-key similarity.')

# ── Visualise attention ───────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Raw Q·Kᵀ scores (before mask + softmax)
raw_scores = (q @ k.transpose(-2,-1))[0].detach()
im0 = axes[0].imshow(raw_scores.numpy(), cmap='RdBu_r')
plt.colorbar(im0, ax=axes[0])
axes[0].set_title('Q·Kᵀ (raw scores)\nbefore mask/scale/softmax', fontweight='bold')

# After scaling but before mask
scaled_scores = (q @ k.transpose(-2,-1) * scale)[0].detach()
im1 = axes[1].imshow(scaled_scores.numpy(), cmap='RdBu_r')
plt.colorbar(im1, ax=axes[1])
axes[1].set_title(f'After ÷√{head_size}\n(variance controlled)', fontweight='bold')

# Causal mask
axes[2].imshow(mask.numpy(), cmap='Blues')
axes[2].set_title('Causal Mask\n(1=attend, 0=blocked)', fontweight='bold')
for i in range(T):
    for j in range(T):
        axes[2].text(j, i, '✓' if i>=j else '✗', ha='center', va='center',
                    color='white' if i<j else 'navy', fontsize=9)

# Final attention weights
im3 = axes[3].imshow(attn[0].detach().numpy(), cmap='Oranges', vmin=0, vmax=1)
plt.colorbar(im3, ax=axes[3])
axes[3].set_title('Final Attention Weights\n(after softmax + mask)', fontweight='bold')

for ax in axes:
    ax.set_xlabel('Key position'); ax.set_ylabel('Query position')

plt.suptitle('Self-Attention: Step-by-Step Construction', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
# 🔷 PHASE 3 — Full GPT Architecture

We now assemble all components into the complete nanoGPT.

```
Head                — single attention head (Q, K, V)
  ↓
MultiHeadAttention  — N parallel heads, concatenated
  ↓
FeedForward         — position-wise MLP (think about gathered context)
  ↓
Block               — Attn + FFN with LayerNorm + residual connections
  ↓
NanoGPT             — token embed + pos embed + N×Block + LM head
```
---

In [ ]:
# ============================================================
#  COMPONENT 1 — Single Attention Head
# ============================================================

class Head(nn.Module):
    """
    One head of causal self-attention.

    Flow:
      x (B,T,C) → Q,K,V projections
               → scaled dot-product attention scores
               → causal mask (future = -inf)
               → softmax → attention weights
               → weighted sum of V
               → output (B, T, head_size)
    """

    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        # Q, K, V projection matrices — no bias (GPT-2 convention)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.key   = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        # tril is a fixed buffer (not trained) — causal mask
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B, T, C = x.shape                   # C = N_EMBD

        # Project to Q, K, V spaces
        q = self.query(x)                   # (B, T, head_size)
        k = self.key(x)                     # (B, T, head_size)
        v = self.value(x)                   # (B, T, head_size)

        # Scaled dot-product attention scores
        scale = self.head_size ** -0.5      # 1/√head_size
        wei   = q @ k.transpose(-2, -1) * scale  # (B, T, T)

        # Causal mask: token at position t cannot see t+1, t+2, ...
        wei   = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # Softmax over keys — rows sum to 1
        wei   = F.softmax(wei, dim=-1)      # (B, T, T)

        # Dropout on attention weights (randomly drop some connections)
        wei   = self.dropout(wei)

        # Weighted aggregation of values
        out   = wei @ v                     # (B, T, head_size)
        return out


# Quick test
test_head = Head(head_size=HEAD_SIZE).to(DEVICE)
test_x    = torch.randn(BATCH_SIZE, BLOCK_SIZE, N_EMBD, device=DEVICE)
test_out  = test_head(test_x)
print(f'Head input  shape: {test_x.shape}')
print(f'Head output shape: {test_out.shape}   ← (B, T, head_size)')
print(f'Head params: {sum(p.numel() for p in test_head.parameters()):,}')

In [ ]:
# ============================================================
#  COMPONENT 2 — Multi-Head Attention
# ============================================================
#
# WHY MULTIPLE HEADS?
# Different heads can learn different types of relationships:
#   Head 1: short-range dependencies (adjacent chars/words)
#   Head 2: subject-verb agreement
#   Head 3: coreference (she → Alice)
#   Head 4: semantic roles
#
# All heads run in PARALLEL (same compute cost as one fat head,
# but richer representations).
#
# After concatenation: N_HEAD × head_size = N_EMBD (by design).
# We project back to N_EMBD with Wₒ.

class MultiHeadAttention(nn.Module):
    """N_HEAD parallel attention heads, results concatenated and projected."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # Output projection: maps from num_heads*head_size → N_EMBD
        self.proj    = nn.Linear(num_heads * head_size, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        # Run each head independently, concat on channel dim
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, N_HEAD*head_size)
        out = self.dropout(self.proj(out))                    # (B, T, N_EMBD)
        return out


# ============================================================
#  COMPONENT 3 — Feed-Forward Network (MLP)
# ============================================================
#
# After attention, each token has GATHERED context from others.
# Now it needs to THINK about that context independently.
#
# The FFN is applied to each token position INDEPENDENTLY.
# It's position-wise: the same MLP is applied to every (b, t) slice.
#
# Hidden dim = 4 × N_EMBD   (from the original Transformer paper)
# Activation = GELU         (GPT-2 uses this instead of ReLU)
#   GELU(x) = x · Φ(x)     (smoother than ReLU, small gradient for x<0)

class FeedForward(nn.Module):
    """Two-layer MLP with GELU activation, applied position-wise."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # expand by 4×
            nn.GELU(),                         # smooth non-linearity
            nn.Linear(4 * n_embd, n_embd),    # project back
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)   # (B, T, N_EMBD) → (B, T, N_EMBD)


# Test components
mha  = MultiHeadAttention(N_HEAD, HEAD_SIZE).to(DEVICE)
ffn  = FeedForward(N_EMBD).to(DEVICE)
test_x = torch.randn(BATCH_SIZE, BLOCK_SIZE, N_EMBD, device=DEVICE)

print('Component shapes:')
print(f'  MHA  input:  {test_x.shape}')
print(f'  MHA  output: {mha(test_x).shape}   ← same shape (B, T, C)')
print(f'  FFN  output: {ffn(test_x).shape}   ← same shape (B, T, C)')
print(f'\n  MHA  params: {sum(p.numel() for p in mha.parameters()):,}')
print(f'  FFN  params: {sum(p.numel() for p in ffn.parameters()):,}')

In [ ]:
# ============================================================
#  COMPONENT 4 — Transformer Block
# ============================================================
#
# One block = Attention + FFN, each with:
#   1. Pre-LayerNorm  (normalise BEFORE the sublayer — modern GPT style)
#   2. Residual connection  (add input back after sublayer)
#
# Formula: x = x + sublayer(LayerNorm(x))
#
# WHY RESIDUAL CONNECTIONS?
#   - Gradient highway: gradient flows directly through '+'
#   - Allows very deep networks (GPT-3 has 96 layers!)
#   - At init: sublayer ≈ 0, so x ≈ x (identity mapping)
#   - Training gradually 'activates' the sublayer
#
# WHY LAYERNORM?
#   - Normalises each token's embedding to mean=0, std=1
#   - Makes training stable across different embedding scales
#   - Computed per-token (unlike BatchNorm which is per-feature)
#
# PRE-LN vs POST-LN:
#   Original Transformer: Post-LN (less stable)
#   GPT-2 and modern LLMs: Pre-LN (more stable, better gradients)

class Block(nn.Module):
    """One Transformer block: Pre-LN + Attention + Residual + Pre-LN + FFN + Residual."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size   = n_embd // n_head
        self.sa     = MultiHeadAttention(n_head, head_size)  # communication
        self.ffwd   = FeedForward(n_embd)                    # computation
        self.ln1    = nn.LayerNorm(n_embd)   # norm before attention
        self.ln2    = nn.LayerNorm(n_embd)   # norm before FFN

    def forward(self, x):
        # Attention branch: communicate between tokens
        x = x + self.sa(self.ln1(x))     # residual: x = x + attn(LN(x))
        # FFN branch: each token processes its own representation
        x = x + self.ffwd(self.ln2(x))   # residual: x = x + ffn(LN(x))
        return x


# Demonstrate LayerNorm
print('=== LayerNorm Demonstration ===')
x_before = torch.randn(2, 5, N_EMBD) * 10 + 5  # arbitrary mean/std
ln_demo  = nn.LayerNorm(N_EMBD)
x_after  = ln_demo(x_before)
print(f'Before LN — mean: {x_before[0,0].mean():.3f}, std: {x_before[0,0].std():.3f}')
print(f'After  LN — mean: {x_after[0,0].mean():.3f}, std: {x_after[0,0].std():.3f}')
print('LayerNorm normalises each token\'s embedding to ~N(0,1), then applies learnable γ,β.')

# Test block
block    = Block(N_EMBD, N_HEAD).to(DEVICE)
test_out = block(test_x)
print(f'\nBlock input:  {test_x.shape}')
print(f'Block output: {test_out.shape}   ← shape preserved')
print(f'Block params: {sum(p.numel() for p in block.parameters()):,}')

In [ ]:
# ============================================================
#  COMPONENT 5 — Complete NanoGPT
# ============================================================
#
# Architecture summary:
#
#  1. Token Embedding:     vocab_size → N_EMBD  (lookup table)
#  2. Positional Embed:    BLOCK_SIZE → N_EMBD  (learned position signals)
#     x = tok_emb + pos_emb  ← ADD (not concat) them
#  3. Dropout on embeddings
#  4. N_LAYER × Block      (self-attention + FFN + LayerNorm)
#  5. Final LayerNorm
#  6. Language Model Head: N_EMBD → vocab_size  (what token comes next?)
#
# WEIGHT INITIALISATION:
#   Linear layers: N(0, 0.02) — from GPT-2 paper
#   Embedding: N(0, 0.02)
#   Residual projection: N(0, 0.02/√(2*N_LAYER)) — scaled down
#   (Prevents residual stream from growing too large with depth)

class NanoGPT(nn.Module):
    """Complete GPT-style language model — Karpathy's nanoGPT."""

    def __init__(self):
        super().__init__()

        # ── Embeddings ──────────────────────────────────────────
        # Token embedding: each of 65 chars → a C-dim vector
        self.tok_emb = nn.Embedding(VOCAB_SIZE, N_EMBD)
        # Positional embedding: position 0..T-1 → a C-dim vector
        # LEARNED (not sinusoidal) — simpler and works as well for short contexts
        self.pos_emb = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop    = nn.Dropout(DROPOUT)

        # ── Transformer blocks ──────────────────────────────────
        self.blocks  = nn.Sequential(*[Block(N_EMBD, N_HEAD) for _ in range(N_LAYER)])

        # ── Final layer norm + output head ──────────────────────
        self.ln_f    = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, VOCAB_SIZE)  # project → vocab logits

        # ── Weight initialisation ───────────────────────────────
        self.apply(self._init_weights)
        # Special scaled init for residual projections (Karpathy trick)
        for pn, p in self.named_parameters():
            if pn.endswith('proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * N_LAYER))

    def _init_weights(self, module):
        """GPT-2 style: N(0, 0.02) for linear+embedding, zeros for biases."""
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        """
        Args:
            idx     : (B, T)  — input token IDs
            targets : (B, T)  — target token IDs (optional, for loss)
        Returns:
            logits  : (B, T, VOCAB_SIZE)
            loss    : scalar cross-entropy (or None if no targets)
        """
        B, T = idx.shape
        assert T <= BLOCK_SIZE, f'Sequence {T} > block_size {BLOCK_SIZE}'

        # ── Embed tokens + positions ─────────────────────────────
        tok  = self.tok_emb(idx)                              # (B, T, C)
        pos  = self.pos_emb(torch.arange(T, device=DEVICE))  # (T, C)
        x    = self.drop(tok + pos)                           # (B, T, C)

        # ── Pass through transformer blocks ──────────────────────
        x    = self.blocks(x)   # (B, T, C)

        # ── Final norm + project to vocab ────────────────────────
        x      = self.ln_f(x)         # (B, T, C)
        logits = self.lm_head(x)      # (B, T, VOCAB_SIZE)

        if targets is None:
            return logits, None

        # ── Compute cross-entropy loss ────────────────────────────
        B, T, C = logits.shape
        loss = F.cross_entropy(
            logits.view(B*T, C),
            targets.view(B*T)
        )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens,
                 temperature=1.0, top_k=None, top_p=None):
        """
        Auto-regressive generation.

        Args:
            idx:           (B, T) — seed token IDs
            max_new_tokens: how many tokens to generate
            temperature:   > 1 → more random, < 1 → more focused
            top_k:         restrict to top-k tokens at each step
            top_p:         nucleus sampling — smallest set with p prob mass
        """
        self.eval()
        for _ in range(max_new_tokens):
            # Crop context to BLOCK_SIZE (the model has a fixed window)
            idx_cond  = idx[:, -BLOCK_SIZE:]
            logits, _ = self(idx_cond)           # (B, T, V)
            logits    = logits[:, -1, :]         # take last step: (B, V)
            logits    = logits / temperature     # apply temperature

            # ── Top-k filtering ──────────────────────────────────
            if top_k is not None:
                k          = min(top_k, logits.size(-1))
                topk_vals, _ = torch.topk(logits, k)
                # Zero out everything below the k-th value
                logits[logits < topk_vals[:, [-1]]] = float('-inf')

            # ── Top-p (nucleus) filtering ─────────────────────────
            if top_p is not None:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                # Remove tokens beyond the nucleus (cumulative prob > top_p)
                to_remove = cum_probs - F.softmax(sorted_logits, dim=-1) > top_p
                sorted_logits[to_remove] = float('-inf')
                logits = torch.zeros_like(logits).scatter(
                    1, sorted_idx, sorted_logits)

            probs  = F.softmax(logits, dim=-1)         # (B, V)
            next_t = torch.multinomial(probs, 1)        # (B, 1) — sample
            idx    = torch.cat([idx, next_t], dim=1)   # (B, T+1)

        self.train()
        return idx

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())


# ── Instantiate ───────────────────────────────────────────────
model = NanoGPT().to(DEVICE)
total = model.get_num_params()

print('=' * 55)
print('         NanoGPT Architecture Summary')
print('=' * 55)
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f'  {name:<12} {str(module.__class__.__name__):<25} {n:>10,} params')
print('=' * 55)
print(f'  TOTAL                                    {total:>10,} params ({total/1e6:.2f}M)')
print()

# ── Before training: check initial loss ─────────────────────
# At random init: loss should be ≈ log(vocab_size) = log(65) ≈ 4.17
x0, y0 = get_batch('train')
_, init_loss = model(x0, y0)
print(f'Initial loss: {init_loss.item():.4f}   (expected ≈ {math.log(VOCAB_SIZE):.4f})')
assert abs(init_loss.item() - math.log(VOCAB_SIZE)) < 0.5, 'Weight init problem!'
print('✅ Weight initialisation correct!')

# ── Sample before training (should be garbage) ───────────────
ctx     = torch.zeros((1,1), dtype=torch.long, device=DEVICE)
pre_text = decode(model.generate(ctx, 100)[0].tolist())
print(f'\nPre-training output (random): {repr(pre_text[:80])}')

---
# 🔷 PHASE 4 — Training, Loss Curves & Text Generation
---

In [ ]:
# ============================================================
#  TRAINING LOOP — AdamW + LR Schedule + Gradient Clipping
# ============================================================
#
# KEY TRAINING DECISIONS:
#
# 1. ADAMW OPTIMIZER
#    Adam = momentum + adaptive learning rates per parameter
#    W = 'weight decay' added correctly (L2 on weights, NOT biases)
#    weight_decay=0.1 for weight matrices, 0.0 for biases/LayerNorm
#
# 2. COSINE LR SCHEDULE
#    Linear warmup (0 → peak LR over 'warmup' steps)
#    Then cosine decay to min_lr
#    Why warmup? At init, gradients are noisy — start conservative.
#
# 3. GRADIENT CLIPPING
#    max_norm=1.0: if gradient vector norm > 1.0, scale it down.
#    Prevents 'exploding gradients' — critical for deep transformers.
#
# 4. set_to_none=True in zero_grad()
#    Frees gradient memory (slightly faster than zero-ing).

# ── Separate param groups for weight decay ────────────────────
# We apply weight decay ONLY to weight matrices (dim >= 2).
# NOT to biases, LayerNorm scale/shift, embeddings.
# (Karpathy: 'biases are not weight-decayed in the original GPT')
decay_params    = [p for n, p in model.named_parameters() if p.dim() >= 2]
no_decay_params = [p for n, p in model.named_parameters() if p.dim() <  2]

optimizer = torch.optim.AdamW([
    {'params': decay_params,    'weight_decay': 0.1},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=LEARNING_RATE, betas=(0.9, 0.95), eps=1e-8)

print(f'Decay params    : {len(decay_params)} tensors, '
      f'{sum(p.numel() for p in decay_params):,} elements')
print(f'No-decay params : {len(no_decay_params)} tensors, '
      f'{sum(p.numel() for p in no_decay_params):,} elements')


# ── LR Schedule: warmup + cosine decay ───────────────────────
WARMUP_ITERS = 200
MIN_LR       = LEARNING_RATE / 10

def get_lr(step):
    # 1. Linear warmup
    if step < WARMUP_ITERS:
        return LEARNING_RATE * step / WARMUP_ITERS
    # 2. After max_iters: return min_lr
    if step > MAX_ITERS:
        return MIN_LR
    # 3. Cosine decay between warmup and max_iters
    progress = (step - WARMUP_ITERS) / (MAX_ITERS - WARMUP_ITERS)
    return MIN_LR + 0.5 * (LEARNING_RATE - MIN_LR) * (1 + math.cos(math.pi * progress))


# ── Visualise LR schedule ─────────────────────────────────────
steps_vis = range(MAX_ITERS)
lrs_vis   = [get_lr(s) for s in steps_vis]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(steps_vis, lrs_vis, 'steelblue', lw=2)
ax.axvline(WARMUP_ITERS, color='red', ls='--', lw=1, label=f'Warmup ends ({WARMUP_ITERS})')
ax.axhline(MIN_LR, color='green', ls=':', lw=1, label=f'Min LR ({MIN_LR:.1e})')
ax.set_xlabel('Training Step'); ax.set_ylabel('Learning Rate')
ax.set_title('Cosine LR Schedule with Linear Warmup', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

print('\nLR at key steps:')
for s in [0, 50, 200, 500, 1000, 2000, 5000]:
    print(f'  step {s:5d}: LR = {get_lr(s):.6f}')

In [ ]:
# ── Main training loop ────────────────────────────────────────
train_log = {'step': [], 'train_loss': [], 'val_loss': [],
              'lr': [], 'grad_norm': []}
best_val_loss = float('inf')
t0 = time.time()

print(f'Training NanoGPT — {MAX_ITERS} steps on {DEVICE}')
print(f'{"Step":>6}  {"Train":>9}  {"Val":>9}  {"PPL":>8}  {"LR":>9}  {"GNorm":>7}')
print('─' * 58)

for step in range(MAX_ITERS):

    # ── Set learning rate ─────────────────────────────────────
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g['lr'] = lr

    # ── Evaluate periodically ─────────────────────────────────
    if step % EVAL_INTERVAL == 0 or step == MAX_ITERS - 1:
        losses = estimate_loss(model)
        ppl    = math.exp(min(losses['val'], 9.0))
        elapsed = time.time() - t0
        tok_per_sec = step * BATCH_SIZE * BLOCK_SIZE / max(elapsed, 1)

        train_log['step'].append(step)
        train_log['train_loss'].append(losses['train'])
        train_log['val_loss'].append(losses['val'])
        train_log['lr'].append(lr)

        flag = ' ← best' if losses['val'] < best_val_loss else ''
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']

        gn = train_log['grad_norm'][-1] if train_log['grad_norm'] else 0.0
        print(f'{step:>6}  {losses["train"]:>9.4f}  {losses["val"]:>9.4f}  '
              f'{ppl:>8.2f}  {lr:>9.2e}  {gn:>7.3f}{flag}')

    # ── Forward + backward ────────────────────────────────────
    xb, yb  = get_batch('train')
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)   # clear old grads
    loss.backward()                          # compute gradients

    # Gradient clipping — prevents exploding gradients
    gn = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0).item()
    train_log['grad_norm'].append(gn)

    optimizer.step()                         # update parameters

total_time = time.time() - t0
print('─' * 58)
print(f'Done! {total_time:.0f}s  |  Best val loss: {best_val_loss:.4f}  '
      f'|  PPL: {math.exp(best_val_loss):.2f}')
print(f'Tokens/sec: {MAX_ITERS * BATCH_SIZE * BLOCK_SIZE / total_time:,.0f}')

In [ ]:
# ── Plot complete training dashboard ─────────────────────────

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
ax4 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1])
ax6 = fig.add_subplot(gs[1, 2])

steps = train_log['step']

# ── 1. Loss curves ────────────────────────────────────────────
ax1.plot(steps, train_log['train_loss'], 'b-o', ms=4, lw=2, label='Train')
ax1.plot(steps, train_log['val_loss'],   'r-o', ms=4, lw=2, label='Val')
ax1.axhline(math.log(VOCAB_SIZE), color='gray', ls=':', lw=1, label=f'Random ({math.log(VOCAB_SIZE):.2f})')
ax1.set_xlabel('Step'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training & Validation Loss', fontweight='bold')
ax1.legend()

# ── 2. Perplexity ──────────────────────────────────────────────
ppl_train = [math.exp(min(l, 9)) for l in train_log['train_loss']]
ppl_val   = [math.exp(min(l, 9)) for l in train_log['val_loss']]
ax2.plot(steps, ppl_train, 'b-o', ms=4, lw=2, label='Train PPL')
ax2.plot(steps, ppl_val,   'r-o', ms=4, lw=2, label='Val PPL')
ax2.set_xlabel('Step'); ax2.set_ylabel('Perplexity')
ax2.set_title('Perplexity = eˡᵒˢˢ\n(how surprised the model is)', fontweight='bold')
ax2.legend()

# ── 3. Gap (overfitting indicator) ────────────────────────────
gap = [v - t for t, v in zip(train_log['train_loss'], train_log['val_loss'])]
ax3.plot(steps, gap, 'purple', lw=2)
ax3.axhline(0, color='black', lw=0.8, ls='--')
ax3.fill_between(steps, 0, gap, alpha=0.2, color='purple')
ax3.set_xlabel('Step'); ax3.set_ylabel('Val Loss − Train Loss')
ax3.set_title('Generalisation Gap\n(0 = perfect, ↑ = overfitting)', fontweight='bold')

# ── 4. Learning rate ──────────────────────────────────────────
ax4.plot(steps, train_log['lr'], 'green', lw=2)
ax4.set_xlabel('Step'); ax4.set_ylabel('Learning Rate')
ax4.set_title('LR Schedule\n(warmup → cosine decay)', fontweight='bold')

# ── 5. Gradient norms ─────────────────────────────────────────
gn_smooth = np.convolve(train_log['grad_norm'], np.ones(50)/50, 'valid')
ax5.plot(train_log['grad_norm'], alpha=0.2, color='orange', lw=0.5)
ax5.plot(np.arange(len(gn_smooth)) + 25, gn_smooth, 'orange', lw=2, label='Smoothed')
ax5.axhline(1.0, color='red', ls='--', lw=1, label='Clip threshold')
ax5.set_xlabel('Step'); ax5.set_ylabel('Gradient Norm')
ax5.set_title('Gradient Norms\n(clipped at 1.0)', fontweight='bold')
ax5.legend()

# ── 6. Bigram vs nanoGPT comparison ──────────────────────────
models_comp = ['Random\nguess', 'Bigram\n(our)', 'NanoGPT\n(ours)', 'GPT-2\nSmall*']
losses_comp = [math.log(VOCAB_SIZE), bigram_losses['val'][-1],
                best_val_loss, 1.33]
colors_comp = ['gray', '#e74c3c', '#2ecc71', '#3498db']
bars = ax6.bar(models_comp, losses_comp, color=colors_comp, edgecolor='white', width=0.5)
for bar, l in zip(bars, losses_comp):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
              f'{l:.2f}', ha='center', fontweight='bold', fontsize=10)
ax6.set_ylabel('Validation Loss'); ax6.set_ylim(0, max(losses_comp)*1.2)
ax6.set_title('Model Comparison\n(*on different dataset)', fontweight='bold')
ax6.text(0.98, 0.02, 'Lower is better', transform=ax6.transAxes,
          ha='right', va='bottom', fontsize=9, color='gray')

fig.suptitle('NanoGPT Training Dashboard', fontsize=15, fontweight='bold', y=0.98)
plt.savefig('/tmp/nanogpt_training.png', dpi=120, bbox_inches='tight')
plt.show()
print('Dashboard saved to /tmp/nanogpt_training.png')

In [ ]:
# ============================================================
#  TEXT GENERATION — All Sampling Strategies
# ============================================================
#
# How the model generates text:
# 1. Encode prompt → token IDs
# 2. Forward pass → logits for next token (vocab_size values)
# 3. Apply sampling strategy → probability distribution
# 4. Sample one token from distribution
# 5. Append to sequence, repeat
#
# SAMPLING STRATEGIES:
#
#  GREEDY (T→0): always pick the argmax
#    → deterministic, repetitive, safe
#
#  TEMPERATURE (T>1): divide logits by T before softmax
#    → T<1: sharper distribution, more predictable
#    → T>1: flatter distribution, more random/creative
#    → T=1: pure model distribution
#
#  TOP-K: zero out all logits except the top k
#    → Cuts off the long tail of improbable tokens
#    → k=1 is greedy, k=vocab_size is unconstrained
#
#  TOP-P (Nucleus): keep smallest set of tokens that sum to p
#    → Adaptive: dynamically adjusts how many tokens to consider
#    → If model is confident: small nucleus (few tokens)
#    → If uncertain: large nucleus (many tokens)
#    → Holtzman et al. 2020: most popular for open-ended generation

def gen(prompt='\n', max_tokens=250, **kwargs):
    """Generate text from a prompt string."""
    enc = torch.tensor(encode(prompt), dtype=torch.long, device=DEVICE).unsqueeze(0)
    out = model.generate(enc, max_tokens, **kwargs)
    return decode(out[0].tolist())


print('=' * 70)
print('TEXT GENERATION — SAMPLING STRATEGY COMPARISON')
print('=' * 70)

strategies = [
    ('Greedy (T=0.1, top_k=1)',        dict(temperature=0.1,  top_k=1)),
    ('Conservative (T=0.5, top_k=20)', dict(temperature=0.5,  top_k=20)),
    ('Balanced (T=0.8, top_k=40)',     dict(temperature=0.8,  top_k=40)),
    ('Creative (T=1.2)',               dict(temperature=1.2)),
    ('Nucleus p=0.9 (top_p)',         dict(temperature=1.0,  top_p=0.9)),
    ('Nucleus p=0.95, T=0.8',         dict(temperature=0.8,  top_p=0.95)),
]

for name, kwargs in strategies:
    print(f'\n▶ {name}')
    print('─' * 60)
    print(gen('KING RICHARD:\n', max_tokens=180, **kwargs))
    print()

print('=' * 70)
print('PROMPT-CONDITIONED GENERATION')
print('=' * 70)

prompts = [
    ('Classic quote',  'To be, or not to be, that is the'),
    ('Character',      'HAMLET:\n'),
    ('Stage direction','Enter ROMEO and JULIET:\n'),
    ('Dialogue',       'FIRST CITIZEN:\nBefore we proceed'),
]
for label, prompt in prompts:
    print(f'\n▶ {label} | Prompt: {repr(prompt)}')
    print('─' * 60)
    print(gen(prompt, max_tokens=200, temperature=0.8, top_k=40, top_p=0.95))

In [ ]:
# ============================================================
#  ATTENTION PATTERN VISUALISATION
#  What did each attention head learn?
# ============================================================

@torch.no_grad()
def extract_attention(model, text, layer_idx=0):
    """
    Extract attention weight matrices from a specified layer.
    Returns: list of (T, T) arrays, one per head.
    """
    model.eval()
    tokens = encode(text[:BLOCK_SIZE])
    T_loc  = len(tokens)
    idx    = torch.tensor(tokens, dtype=torch.long, device=DEVICE).unsqueeze(0)

    # Build up activations layer by layer
    tok = model.tok_emb(idx)
    pos = model.pos_emb(torch.arange(T_loc, device=DEVICE))
    x   = model.drop(tok + pos)

    # Pass through earlier blocks
    for i in range(layer_idx):
        x = model.blocks[i](x)

    # Extract attention from target block
    block  = model.blocks[layer_idx]
    x_norm = block.ln1(x)
    tril_m = torch.tril(torch.ones(T_loc, T_loc, device=DEVICE))

    attn_maps = []
    for head in block.sa.heads:
        q   = head.query(x_norm)     # (1, T, hs)
        k   = head.key(x_norm)
        hs  = q.shape[-1]
        wei = q @ k.transpose(-2,-1) * (hs ** -0.5)   # (1, T, T)
        wei = wei.masked_fill(tril_m == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        attn_maps.append(wei[0].cpu().numpy())         # (T, T)

    return attn_maps, tokens


# Choose a meaningful text snippet
sample_text = 'To be or not to be that is the question whether'
char_labels = list(sample_text[:BLOCK_SIZE])

# Visualise attention for each layer
fig, all_axes = plt.subplots(N_LAYER, N_HEAD, figsize=(4*N_HEAD, 3.5*N_LAYER))
if N_LAYER == 1: all_axes = all_axes[np.newaxis, :]

for layer_i in range(N_LAYER):
    maps, tokens = extract_attention(model, sample_text, layer_i)
    T_plot = len(tokens)

    for head_j, (ax, amap) in enumerate(zip(all_axes[layer_i], maps)):
        # Entropy of each row (how spread out is the attention?)
        eps    = 1e-9
        ent    = -(amap * np.log(amap + eps)).sum(axis=1)  # (T,)
        avg_ent = ent.mean()

        im = ax.imshow(amap[:T_plot, :T_plot], cmap='Blues',
                        aspect='auto', interpolation='nearest', vmin=0)

        ax.set_xticks(range(T_plot))
        ax.set_yticks(range(T_plot))
        ax.set_xticklabels(char_labels[:T_plot], fontsize=6, rotation=90)
        ax.set_yticklabels(char_labels[:T_plot], fontsize=6)
        ax.set_title(f'L{layer_i} H{head_j}  ent={avg_ent:.2f}',
                      fontsize=9, fontweight='bold')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(f'Learned Attention Patterns — All Layers & Heads\n'
              f'Text: "{sample_text[:40]}..."\n'
              f'Row = query token, Col = key it attends to. '
              f'Lower triangle = causal. Higher entropy = more spread.',
              fontsize=10, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/tmp/nanogpt_attention.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved to /tmp/nanogpt_attention.png')

---
# 🔷 PHASE 5 — Deep Dives: Scaling, Modern Tricks, GPT Family
---

In [ ]:
# ============================================================
#  SCALING LAWS — How loss scales with model/data/compute
# ============================================================
#
# Kaplan et al. (OpenAI, 2020): "Scaling Laws for Neural LMs"
# Hoffmann et al. (DeepMind, 2022): "Training Compute-Optimal LLMs" (Chinchilla)
#
# KEY FINDING: Loss decreases as a power law with:
#   - Model size (# parameters N)
#   - Dataset size (# tokens D)
#   - Compute (FLOPs C = 6ND for training)
#
# L(N) ≈ (Nc/N)^αN + L∞
#
# CHINCHILLA RULE (optimal compute allocation):
#   Optimal tokens = 20 × parameters
#   GPT-3 (175B) should train on 3.5T tokens
#   LLaMA-2 (70B) trained on 2T tokens (closer to optimal)

import matplotlib.pyplot as plt
import numpy as np

# ── Scaling law simulation ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Real data points (approximate, from literature)
gpt_family = [
    ('GPT-2 Small',  124e6,   1.33, '#3498db'),
    ('GPT-2 Medium', 355e6,   1.26, '#2980b9'),
    ('GPT-2 Large',  774e6,   1.20, '#1a5276'),
    ('GPT-2 XL',     1.5e9,   1.15, '#154360'),
    ('GPT-3',        175e9,   0.98, '#8e44ad'),
    ('LLaMA 7B',     7e9,     1.05, '#e67e22'),
    ('LLaMA 70B',    70e9,    0.92, '#d35400'),
]

our_models = [
    ('Bigram',   VOCAB_SIZE**2,   bigram_losses['val'][-1],  '#e74c3c'),
    ('NanoGPT',  model.get_num_params(), best_val_loss, '#2ecc71'),
]

# Plot 1: Loss vs Parameters
for name, params, loss, color in gpt_family:
    axes[0].scatter(params, loss, s=100, color=color, zorder=5)
    if 'GPT-2 Small' in name or 'GPT-3' in name or 'LLaMA 70' in name:
        axes[0].annotate(name, (params, loss), fontsize=7,
                          xytext=(10, 5), textcoords='offset points')
for name, params, loss, color in our_models:
    axes[0].scatter(params, loss, s=150, color=color, marker='*', zorder=6,
                    edgecolors='black', linewidth=0.8, label=name)
    axes[0].annotate(name, (params, loss), fontsize=8, fontweight='bold',
                      xytext=(10, -12), textcoords='offset points', color=color)

# Fit power law trend through GPT family
p_arr = np.array([x[1] for x in gpt_family])
l_arr = np.array([x[2] for x in gpt_family])
coeffs = np.polyfit(np.log10(p_arr), l_arr, 1)
p_line = np.logspace(8, 11, 100)
axes[0].plot(p_line, np.polyval(coeffs, np.log10(p_line)), 'k--', lw=1, alpha=0.5)

axes[0].set_xscale('log')
axes[0].set_xlabel('Parameters (log scale)', fontweight='bold')
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Scaling Law: Loss vs Parameters\n(our models marked with ★)',
                   fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper right', fontsize=8)

# Plot 2: Chinchilla compute-optimal ratio
params_range = np.logspace(8, 12, 100)
optimal_tokens = 20 * params_range   # Chinchilla rule
axes[1].loglog(params_range, optimal_tokens, 'blue', lw=2.5, label='Chinchilla optimal (20×)')
axes[1].loglog(params_range, 300e9 * np.ones_like(params_range), 'r--', lw=1.5,
               label='GPT-3 training tokens (300B)')
axes[1].scatter([175e9], [300e9], s=150, color='purple', zorder=5, label='GPT-3')
axes[1].scatter([70e9],  [2e12],  s=150, color='orange', zorder=5, label='LLaMA-2 70B')
axes[1].set_xlabel('Model Parameters', fontweight='bold')
axes[1].set_ylabel('Training Tokens')
axes[1].set_title('Chinchilla Optimal Compute\n(20 tokens per parameter)', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Plot 3: Context length evolution
years   = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
ctx_len = [512,  1024, 2048, 2048, 4096, 128000, 1000000]
models_ = ['BERT', 'GPT-2', 'GPT-3', 'GPT-3.5', 'LLaMA', 'GPT-4', 'Gemini 1.5']
axes[2].semilogy(years, ctx_len, 'o-', color='steelblue', lw=2.5, ms=8)
for yr, ctx, name in zip(years, ctx_len, models_):
    axes[2].annotate(f'{name}\n{ctx:,}', (yr, ctx), fontsize=7,
                      xytext=(5, 8), textcoords='offset points')
axes[2].set_xlabel('Year'); axes[2].set_ylabel('Context Length (tokens, log)')
axes[2].set_title('Context Length Evolution\n(GPT-4 to Gemini: 500× increase)', fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Scaling Laws & GPT Family Evolution', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  MODERN GPT TRICKS — Visualised
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
x_plot = np.linspace(-4, 4, 400)

# ── 1. Activation Functions ───────────────────────────────────
ax = axes[0, 0]
acts = {
    'ReLU (old)':   np.maximum(0, x_plot),
    'GELU (GPT-2)': x_plot * 0.5*(1 + np.tanh(np.sqrt(2/np.pi)*(x_plot + 0.044715*x_plot**3))),
    'SiLU/Swish':   x_plot / (1 + np.exp(-x_plot)),
    'SoLU':         x_plot * np.exp(-x_plot**2/2) / np.sqrt(2*np.pi),
}
for name, vals in acts.items():
    ax.plot(x_plot, vals, lw=2, label=name)
ax.axhline(0, color='gray', lw=0.8, ls='--'); ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_title('Activation Functions\n(GELU is default in GPT-2+)', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(-4, 4); ax.grid(True, alpha=0.3)

# ── 2. Pre-LN vs Post-LN ─────────────────────────────────────
ax = axes[0, 1]
depth = np.arange(1, 20)
# Simulated gradient magnitude at layer 1 (normalised)
postln_grad = 1.0 / (1.5 ** (depth-1)) + np.random.RandomState(42).randn(19)*0.03
preln_grad  = 0.8 * np.ones_like(depth) + np.random.RandomState(42).randn(19)*0.05
ax.semilogy(depth, np.maximum(1e-5, postln_grad), 'r-o', ms=6, lw=2, label='Post-LN (original)')
ax.semilogy(depth, preln_grad, 'g-o', ms=6, lw=2, label='Pre-LN (GPT-2, modern)')
ax.set_xlabel('Layer depth'); ax.set_ylabel('Gradient magnitude (log)')
ax.set_title('Pre-LN vs Post-LN\n(Pre-LN keeps gradients healthy at depth)',
              fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# ── 3. Sinusoidal vs Learned Positional Encoding ──────────────
ax = axes[0, 2]
T_pe, D_pe = 64, 64
pe_sin = np.zeros((T_pe, D_pe))
for pos in range(T_pe):
    for i in range(0, D_pe, 2):
        pe_sin[pos, i  ] = np.sin(pos / 10000**(2*i/D_pe))
        pe_sin[pos, i+1] = np.cos(pos / 10000**(2*i/D_pe))
im = ax.imshow(pe_sin, aspect='auto', cmap='RdBu_r')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Embedding Dimension'); ax.set_ylabel('Position')
ax.set_title('Sinusoidal Positional Encoding\n(Original Transformer — nanoGPT uses learned)',
              fontweight='bold')

# ── 4. RoPE (Rotary Position Embedding) ──────────────────────
ax = axes[1, 0]
thetas  = np.logspace(0, -4, 32)   # 32 frequency bands
pos_arr = np.arange(64)
# Show rotation angles for positions 0..63, dims 0..31
angle_matrix = np.outer(pos_arr, thetas)
im2 = ax.imshow(np.sin(angle_matrix).T, aspect='auto', cmap='coolwarm')
plt.colorbar(im2, ax=ax)
ax.set_xlabel('Position'); ax.set_ylabel('Frequency dim')
ax.set_title('RoPE: Rotary Position Encoding\n(Used in LLaMA, Mistral, GPT-4 — better generalisation)',
              fontweight='bold')

# ── 5. Flash Attention memory comparison ─────────────────────
ax = axes[1, 1]
seq_lengths = [128, 256, 512, 1024, 2048, 4096, 8192]
std_mem  = [s**2 * 4 / 1e6 for s in seq_lengths]   # O(T²) bytes
flash_mem= [s    * 4 / 1e6 for s in seq_lengths]   # O(T)  bytes (approx)
ax.semilogy(seq_lengths, std_mem,   'r-o', ms=7, lw=2, label='Standard Attention O(T²)')
ax.semilogy(seq_lengths, flash_mem, 'g-o', ms=7, lw=2, label='Flash Attention O(T)')
ax.set_xlabel('Sequence Length'); ax.set_ylabel('Memory (MB, log scale)')
ax.set_title('Flash Attention Memory Efficiency\n(Dao et al. 2022 — tiled SRAM computation)',
              fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# ── 6. GQA: Grouped Query Attention ──────────────────────────
ax = axes[1, 2]
variants = ['MHA\n(all heads\nhave own K,V)', 'GQA\n(groups share\nK,V)', 'MQA\n(all share\none K,V)']
kv_heads = [8, 2, 1]
colors_gqa = ['#e74c3c', '#f39c12', '#2ecc71']
bars = ax.bar(variants, kv_heads, color=colors_gqa, edgecolor='white', width=0.5)
for bar, n in zip(bars, kv_heads):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f'{n} KV head(s)', ha='center', fontweight='bold')
ax.set_ylabel('Number of K,V heads (for 8 query heads)')
ax.set_title('Multi-Head vs Grouped-Query vs Multi-Query\n(GQA = LLaMA-2/3, Mistral; MQA = early fast models)',
              fontweight='bold')
ax.set_ylim(0, 10)

plt.suptitle('Modern GPT Optimizations', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  GPT FAMILY — Architecture Comparison
# ============================================================

import pandas as pd

family = pd.DataFrame([
    ['NanoGPT (ours)',  N_LAYER, N_HEAD, N_EMBD,   BLOCK_SIZE, model.get_num_params(),    'char-level',  VOCAB_SIZE],
    ['GPT-2 Small',    12,      12,     768,      1024,       124_000_000,               'BPE',         50257],
    ['GPT-2 Medium',   24,      16,     1024,     1024,       355_000_000,               'BPE',         50257],
    ['GPT-2 Large',    36,      20,     1280,     1024,       774_000_000,               'BPE',         50257],
    ['GPT-2 XL',       48,      25,     1600,     1024,       1_500_000_000,             'BPE',         50257],
    ['GPT-3',          96,      96,     12288,    2048,       175_000_000_000,           'BPE',         50257],
    ['LLaMA 7B',       32,      32,     4096,     4096,       7_000_000_000,             'SentencePiece', 32000],
    ['LLaMA 70B',      80,      64,     8192,     4096,       70_000_000_000,            'SentencePiece', 32000],
    ['Mistral 7B',     32,      32,     4096,     32768,      7_000_000_000,             'SentencePiece', 32000],
], columns=['Model', 'Layers', 'Heads', 'd_model', 'Context', 'Params', 'Tokenizer', 'Vocab'])

family['Params (B)'] = family['Params'].apply(
    lambda x: f'{x/1e9:.3f}B' if x > 1e9 else f'{x/1e6:.1f}M')

display_cols = ['Model', 'Layers', 'Heads', 'd_model', 'Context', 'Params (B)', 'Vocab']
print('GPT Architecture Family:')
print(family[display_cols].to_string(index=False))

# ── Key differences: nanoGPT → GPT-4 ─────────────────────────
differences = [
    ('Scale',           f'400K params → 1T+ params — 2.5 million× more'),
    ('Context',         f'{BLOCK_SIZE} tokens → 128K tokens (GPT-4)'),
    ('Tokenizer',       f'65 chars → 100K BPE tokens (tiktoken)'),
    ('Positional',      'Learned → RoPE (Rotary Position Embedding)'),
    ('Attention',       'Standard → Flash Attention (memory efficient)'),
    ('Heads',           'MHA → GQA (Grouped Query Attention, faster KV cache)'),
    ('FFN',             'GELU → SwiGLU (gated FFN, better performance)'),
    ('Normalization',   'Post/Pre-LN → RMSNorm (simpler, equally effective)'),
    ('Training data',   '1MB Shakespeare → 10T+ tokens, multimodal'),
    ('Post-training',   'None → RLHF / DPO / instruction tuning'),
]

print('\nHow nanoGPT → production LLM:')
print(f'{"Component":<20} {"Change"}')
print('─' * 75)
for component, change in differences:
    print(f'{component:<20} {change}')

In [ ]:
# ============================================================
#  GPT vs BERT — Architecture Diagram
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

def draw_arch(ax, title, mask_type, use_case, color, pre_train_task):
    ax.set_xlim(0, 10); ax.set_ylim(0, 14)
    ax.axis('off')

    def box(x, y, w, h, label, c, fs=9, bold=False):
        rect = plt.Rectangle((x, y), w, h, fc=c, ec='black', lw=1.5, zorder=3)
        ax.add_patch(rect)
        fw = 'bold' if bold else 'normal'
        ax.text(x+w/2, y+h/2, label, ha='center', va='center',
                fontsize=fs, fontweight=fw, wrap=True, zorder=4)

    def arrow(x1, y1, x2, y2):
        ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

    # Input
    box(1, 0.3, 8, 0.9, 'Input Tokens', '#ecf0f1')
    arrow(5, 1.2, 5, 1.8)

    # Embeddings
    box(1, 1.8, 8, 0.9, 'Token + Positional Embeddings', '#d5dbdb')
    arrow(5, 2.7, 5, 3.3)

    # Attention block
    box(1, 3.3, 8, 1.5,
        f'Transformer Block × N\n({mask_type} Attention)',
        color, fs=10, bold=True)
    arrow(5, 4.8, 5, 5.4)

    # Output head
    box(1, 5.4, 8, 0.9, 'Layer Norm', '#d5dbdb')
    arrow(5, 6.3, 5, 6.9)

    box(1, 6.9, 8, 1.2, 'Output Head', '#abebc6', bold=True)

    # Mask visualisation
    if mask_type == 'Causal (one-way)':
        m = np.tril(np.ones((5,5)))
    else:
        m = np.ones((5,5))
    mask_ax = ax.inset_axes([0.65, 0.55, 0.3, 0.2])
    mask_ax.imshow(m, cmap='Blues', aspect='auto')
    mask_ax.set_title('Mask', fontsize=8, fontweight='bold')
    mask_ax.set_xticks([]); mask_ax.set_yticks([])

    # Info box
    info = (f'Pre-training: {pre_train_task}\n'
            f'Use case: {use_case}\n'
            f'Direction: {mask_type.split("(")[0].strip()}')
    ax.text(5, 9.5, info, ha='center', va='center', fontsize=9,
            bbox=dict(fc='lightyellow', ec='orange', lw=1.5, pad=8),
            family='monospace')

    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

draw_arch(axes[0], 'GPT (Decoder-only)',
          'Causal (one-way)', 'Text generation',
          '#aed6f1', 'Next-token prediction')

draw_arch(axes[1], 'BERT (Encoder-only)',
          'Bidirectional (full)', 'Understanding / classification',
          '#a9dfbf', 'Masked language model (MLM)')

plt.suptitle('GPT vs BERT — Architecture Comparison', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

print('KEY DIFFERENCE:')
print('  GPT: Each token can only see PAST tokens (causal mask)')
print('       → Good for GENERATION (text completion)')
print('  BERT: Each token can see ALL tokens (bidirectional)')
print('       → Good for UNDERSTANDING (classification, QA)')

In [ ]:
# ============================================================
#  KARPATHY'S SANITY CHECK: OVERFIT A SINGLE BATCH
# ============================================================
#
# Before training on the full dataset, ALWAYS do this check:
# Take ONE batch, train ONLY on that batch, verify loss → 0.
#
# If the model can't overfit a single batch:
#   - Your loss function is wrong
#   - Your data pipeline has a bug
#   - Model has insufficient capacity
#
# This is Karpathy's Rule #1: 'Overfit one batch first.'

print('=== Karpathy Sanity Check: Overfit one batch ===')

# Fresh small model
tiny = NanoGPT().to(DEVICE)
tiny_opt = torch.optim.AdamW(tiny.parameters(), lr=1e-3)

# Fix ONE batch (don't sample new batches)
x_fix, y_fix = get_batch('train')

overfit_losses = []
for i in range(500):
    _, loss = tiny(x_fix, y_fix)
    tiny_opt.zero_grad(); loss.backward(); tiny_opt.step()
    overfit_losses.append(loss.item())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(overfit_losses, 'steelblue', lw=2)
ax.axhline(0.01, color='green', ls='--', lw=1, label='~0 target')
ax.set_xlabel('Gradient step'); ax.set_ylabel('Loss on fixed batch')
ax.set_title('Sanity Check: Overfit One Batch\n'
             'Loss should reach ~0 if model + data pipeline are correct',
             fontweight='bold')
ax.legend(); ax.set_yscale('log'); plt.tight_layout(); plt.show()

final_overfit = overfit_losses[-1]
print(f'Final overfit loss: {final_overfit:.6f}')
if final_overfit < 0.1:
    print('✅ PASS — Model can overfit. Data pipeline + loss function are correct.')
else:
    print('⚠️  WARNING — Model did not overfit. Check your implementation.')

---
# 📋 Complete NanoGPT Reference Card
---

## Architecture

```
Input IDs (B, T)
      │
      ├─ Token Embedding  (vocab, C)   → (B, T, C)
      ├─ Position Embed   (T_max, C)   → (B, T, C)   [LEARNED]
      └─ x = tok_emb + pos_emb + Dropout
             │
             ▼
    ┌─── × N_LAYER ──────────────────────────────────┐
    │  x = x + MultiHeadAttention(LayerNorm(x))      │
    │          └─ N_HEAD × [Q·Kᵀ/√d → mask → softmax → ×V]
    │  x = x + FeedForward(LayerNorm(x))             │
    │          └─ Linear(C→4C) → GELU → Linear(4C→C) │
    └────────────────────────────────────────────────┘
             │
      LayerNorm → Linear(C, vocab) → logits (B, T, V)
             │
      cross_entropy(logits, targets) → loss
```

## Key Formulas

| Step | Formula |
|---|---|
| Attention scores | `A = Q·Kᵀ / √head_size` |
| Causal mask | `A[i,j] = -∞ if j > i` |
| Attention weights | `W = softmax(A)` |
| Attention output | `out = W · V` |
| Pre-LN residual | `x = x + sublayer(LN(x))` |
| FFN | `FFN(x) = Linear(GELU(Linear(x, 4C)), C)` |
| Loss | `L = CrossEntropy(logits, targets)` |
| Perplexity | `PPL = e^L` |
| Temperature | `p = softmax(logits / T)` |
| AdamW | `θ -= α · (m̂/√v̂ + λ·θ)` |
| Grad clip | `g = g · min(1, max_norm/‖g‖)` |

## Karpathy's Rules of Thumb

1. **Always look at your data** — read actual samples before coding
2. **Overfit one batch first** — if loss doesn't → 0, you have a bug
3. **Check initial loss** — should be `log(vocab_size)` at random init
4. **Train loss ≈ val loss early** — gap = overfitting
5. **Use AdamW, not Adam** — weight decay matters
6. **Gradient clipping** — always for transformers
7. **LR schedule** — warmup + cosine decay
8. **Residuals + pre-LN** = trainable depth; stack blocks freely
9. **Scale compute, not complexity** — bigger model + more data > tricks
10. **Temperature 0.8, top-k 40 or top-p 0.9** — good generation defaults

## Where to Go Next

| Resource | Link |
|---|---|
| Karpathy's nanoGPT repo | github.com/karpathy/nanoGPT |
| "Let's build GPT" lecture | youtube.com/@AndrejKarpathy |
| "Attention Is All You Need" | arxiv.org/abs/1706.03762 |
| GPT-2 paper | openai.com/research/language-unsupervised |
| Chinchilla (scaling) | arxiv.org/abs/2203.15556 |
| Flash Attention | arxiv.org/abs/2205.14135 |
| HuggingFace Transformers | huggingface.co/docs/transformers |
| LLaMA paper | arxiv.org/abs/2302.13971 |

---
*Built from scratch following Andrej Karpathy's nanoGPT.  
Every tensor shape, every design decision, fully explained.*